In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json

import sys

sys.path.append("../")

##################################################################
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
##################################################################

import logging
from src.utils import logging_utils
from src.utils import env_utils
import torch
import transformers

logger = logging.getLogger(__name__)

logging.basicConfig(
    level=logging.DEBUG,
    format=logging_utils.DEFAULT_FORMAT,
    datefmt=logging_utils.DEFAULT_DATEFMT,
    stream=sys.stdout,
)

logger.info(f"{torch.__version__=}, {torch.version.cuda=}")
logger.info(
    f"{torch.cuda.is_available()=}, {torch.cuda.device_count()=}, {torch.cuda.get_device_name()=}"
)
logger.info(f"{transformers.__version__=}")

/disk/u/arnab/miniconda3/envs/goals/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-01-26 12:46:12 __main__ INFO     torch.__version__='2.10.0+cu128', torch.version.cuda='12.8'
2026-01-26 12:46:12 __main__ INFO     torch.cuda.is_available()=True, torch.cuda.device_count()=2, torch.cuda.get_device_name()='NVIDIA A100 80GB PCIe'
2026-01-26 12:46:12 __main__ INFO     transformers.__version__='4.57.6'


In [3]:
from src.models import ModelandTokenizer

# model_key = "allenai/Olmo-3.1-32B-Instruct"
# model_key = "meta-llama/Llama-3.3-70B-Instruct"
model_key = "Qwen/Qwen2.5-Coder-32B"
# model_key = "deepseek-ai/deepseek-coder-33b-base"
# model_key = "openai/gpt-oss-20b"
# model_key = "Qwen/Qwen3-Coder-30B-A3B-Instruct"
# model_key = "allenai/Olmo-3.1-7B-RL-Zero-Code"

mt = ModelandTokenizer(
    model_key=model_key,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)

2026-01-26 12:46:18 src.models WARNING  Qwen/Qwen2.5-Coder-32B not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory
2026-01-26 12:46:18 urllib3.connectionpool DEBUG    Starting new HTTPS connection (1): huggingface.co:443
2026-01-26 12:46:18 urllib3.connectionpool DEBUG    https://huggingface.co:443 "HEAD /Qwen/Qwen2.5-Coder-32B/resolve/main/config.json HTTP/1.1" 307 0
2026-01-26 12:46:18 urllib3.connectionpool DEBUG    https://huggingface.co:443 "HEAD /api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B/2e12b5f7bc878d424d222e224ed40aee564ec45f/config.json HTTP/1.1" 200 0
2026-01-26 12:46:18 urllib3.connectionpool DEBUG    https://huggingface.co:443 "HEAD /Qwen/Qwen2.5-Coder-32B/resolve/main/tokenizer_config.json HTTP/1.1" 307 0
2026-01-26 12:46:18 urllib3.connectionpool DEBUG    https://huggingface.co:443 "HEAD /api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B/2e12b5f7bc878d424d222e224ed40aee564ec45f/tokenizer_config.js

Loading checkpoint shards: 100%|██████████| 14/14 [00:19<00:00,  1.42s/it]

2026-01-26 12:46:39 urllib3.connectionpool DEBUG    https://huggingface.co:443 "HEAD /Qwen/Qwen2.5-Coder-32B/resolve/main/generation_config.json HTTP/1.1" 307 0
2026-01-26 12:46:39 urllib3.connectionpool DEBUG    https://huggingface.co:443 "HEAD /api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B/2e12b5f7bc878d424d222e224ed40aee564ec45f/generation_config.json HTTP/1.1" 200 0


2026-01-26 12:46:39 urllib3.connectionpool DEBUG    https://huggingface.co:443 "HEAD /Qwen/Qwen2.5-Coder-32B/resolve/main/custom_generate/generate.py HTTP/1.1" 404 0
2026-01-26 12:46:39 src.models INFO     loaded model <Qwen/Qwen2.5-Coder-32B> | size: 62492.135 MB | dtype: torch.bfloat16 | device: cuda:0


In [4]:
# from nnsight import LanguageModel

# lm = LanguageModel(
#     model_key,
#     dispatch=True
# )

In [5]:
# from datasets import load_dataset

# ds = load_dataset("gonglinyuan/safim", "block")

In [6]:
# sample = ds["test"][16]
# sample

In [7]:
# print(sample["prompt"])

In [8]:
# print(sample["eval_prompt"])

In [9]:
# instruction = """Instruction: Complete the {{completion}} in the following Python function.\n\n"""
# code = """# the function calculates (a ^ b) mod m for large positive integers a, b, and m.
# def big_mod_exp(a: int, b: int, m: int) -> int:
#     result = 1
#     a = a % m
#     while b > 0:
#         if {{completion}}:
#             result = (result * a) % m
#         b = b >> 1
#         a = (a * a) % m
#     return result

# Answer: 0 =="""

# instruction = """Complete the following Python function.\n\n"""
# code = """Calculate (a ^ b) mod m for large positive integers a, b, and m.
# def big_mod_exp(a: int, b: int, m: int) -> int:
#     result = 1
#     a = a % m
#     while b > 0:
#         if 0 =="""

In [10]:
from rich import print
from rich.table import Table
import json
import warnings
from typing import Any
from rich.console import Console


def evaluate_single_code(
    full_code: str, unit_tests: list, function_name: str = "func"
) -> dict:
    """
    Evaluate a single code snippet against unit tests.

    Returns a dict with:
        - verdict: "PASSED", "FAILED", "SYNTAX_ERROR", or "ERROR"
        - unit_test_results: list of individual test results
        - warnings: list of captured warnings
        - error_message: error details if any
        - full_code: the code that was evaluated
    """
    result = {
        "verdict": None,
        "unit_test_results": [],
        "warnings": [],
        "error_message": "",
        "full_code": full_code,
    }

    try:
        # Create a namespace to execute the code
        namespace = {}

        # Capture warnings during exec
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always")  # Catch all warnings
            exec(full_code, namespace)
            # Store any warnings that occurred
            result["warnings"] = [
                f"{warning.category.__name__}: {warning.message}" for warning in w
            ]

        # Check if function exists
        if function_name not in namespace:
            result["verdict"] = "FAILED"
            result["error_message"] = f"{function_name} function not defined"
            return result

        func = namespace[function_name]

        # Run unit tests
        for input_val, expected in unit_tests:
            test_result = {
                "input": input_val,
                "expected": expected,
                "actual": None,
                "passed": False,
                "error": None,
            }
            try:
                with warnings.catch_warnings(record=True) as w:
                    warnings.simplefilter("always")
                    actual = func(input_val)
                    # Capture warnings from function execution too
                    result["warnings"].extend(
                        [
                            f"{warning.category.__name__}: {warning.message}"
                            for warning in w
                        ]
                    )

                test_result["actual"] = actual
                test_result["passed"] = actual == expected
            except Exception as e:
                test_result["error"] = f"{type(e).__name__}: {e}"

            result["unit_test_results"].append(test_result)

        all_passed = all(t["passed"] for t in result["unit_test_results"])
        result["verdict"] = "PASSED" if all_passed else "FAILED"

    except SyntaxError as e:
        result["verdict"] = "SYNTAX_ERROR"
        result["error_message"] = str(e)
    except Exception as e:
        result["verdict"] = "ERROR"
        result["error_message"] = f"{type(e).__name__}: {e}"

    return result


def run_evaluation(
    code: str,
    prefills: list[str],
    outputs: list[str],
    unit_tests: list,
    function_name: str = "func",
    output_stop_string: str = "</code>",
) -> dict:
    """
    Run evaluation on multiple code completions.

    Args:
        code: The base code template
        prefills: List of prefill strings
        outputs: List of model outputs corresponding to each prefill
        unit_tests: List of (input, expected_output) tuples
        function_name: Name of the function to test
        output_stop_string: String to split output on (to truncate generation)

    Returns a dict with:
        - code: the base code template
        - function_name: name of the function tested
        - unit_tests: the unit tests used
        - results: list of evaluation results for each prefill/output pair
        - summary: dict with pass_count, fail_count, total
    """
    eval_results = {
        "code": code,
        "function_name": function_name,
        "unit_tests": unit_tests,
        "results": [],
        "summary": {
            "pass_count": 0,
            "fail_count": 0,
            "total": len(prefills),
        },
    }

    for prefill, out in zip(prefills, outputs):
        # Clean up the output
        if output_stop_string:
            out = out.split(output_stop_string)[0].rstrip()

        full_code = code + prefill + out

        single_result = evaluate_single_code(full_code, unit_tests, function_name)
        single_result["prefill"] = prefill
        single_result["output"] = out

        eval_results["results"].append(single_result)

        if single_result["verdict"] == "PASSED":
            eval_results["summary"]["pass_count"] += 1
        else:
            eval_results["summary"]["fail_count"] += 1

    return eval_results


def pretty_print_eval_results(eval_results: dict, output_file: str | None = None) -> None:
    """
    Print evaluation results in a nicely formatted table.

    Args:
        eval_results: Output from run_evaluation()
        output_file: Optional file path to write output to
    """
    # Always create a console for building the output
    console = Console(record=True, width=200)  # record=True captures output

    results_table = Table(title="Code Completion Results", show_lines=True)
    results_table.add_column("Prefill", style="white")
    results_table.add_column("Logic", style="white")
    results_table.add_column("Verdict", style="cyan")
    results_table.add_column("Unit Tests", style="white")
    results_table.add_column("Logs", style="dim", overflow="fold")

    code = eval_results["code"]
    function_name = eval_results["function_name"]

    console.print('"""' + code + '"""')

    for result in eval_results["results"]:
        prefill = result["prefill"]
        out = result["output"]
        verdict = result["verdict"]
        warns = result["warnings"]
        error_msg = result["error_message"]

        # Format verdict
        if verdict == "PASSED":
            verdict_msg = "[bright_green]PASSED[/bright_green]"
        else:
            verdict_msg = f"[bright_red]{verdict}[/bright_red]"

        # Format unit test results
        unit_test_strs = []
        for t in result["unit_test_results"]:
            if t["error"]:
                unit_test_strs.append(
                    f"{function_name}({t['input']}) raised {t['error']}"
                )
            elif t["passed"]:
                unit_test_strs.append(f"{function_name}({t['input']})={t['actual']} ✓")
            else:
                unit_test_strs.append(
                    f"{function_name}({t['input']})={t['actual']}, expected {t['expected']} ✗"
                )
        unit_test_str = "\n".join(unit_test_strs)

        # Format logs
        log_parts = []
        if warns:
            log_parts.append("[yellow]" + "\n".join(warns) + "[/yellow]")
        if error_msg:
            log_parts.append(error_msg)
        if verdict not in ["PASSED", "FAILED"]:
            log_parts.append(f"[dim]{result['full_code']}[/dim]")
        log_str = "\n".join(log_parts)

        results_table.add_row(
            f'"{prefill}"',
            f'"[bright_yellow]{prefill}[/bright_yellow][white]{out}[/white]"',
            verdict_msg,
            unit_test_str if verdict != "PASSED" else "",
            log_str,
        )

    # Print summary
    summary = eval_results["summary"]
    console.print(results_table)
    console.print(
        f"\n[bold]Summary:[/bold] {summary['pass_count']}/{summary['total']} passed, {summary['fail_count']}/{summary['total']} failed"
    )

    # Export to file or print to stdout
    if output_file:
        with open(output_file, "w") as f:
            f.write(console.export_text())  # Plain text with table structure preserved

In [13]:
eval_dataset = [
    {
        "instruction": """Write a function in python that checks if a number is odd.\n\n""",
        "code": """def is_odd(n: int) -> bool:
    return True if """,
        "function_name": "is_odd",
        "unit_tests": [
            (1, True),
            (2, False),
            (0, False),
            (15, True),
            (-3, True),
        ],
        "prefills": [
            "",
            "(",
            "0",
            "abs",
            "1",
            "(1",
            "((1",
            "1 +",
            "not",
            "not 1",
            "not 0",
            "bool",
            "bool(",
            "n &",
            "(n &",
        ],
    }
]

# with open("../sample_eval.json", "r") as f:
#     eval_dataset = json.load(f)
# print(len(eval_dataset))

sample_idx = 0
print(eval_dataset[sample_idx])

{
    'instruction': 'Write a function in python that checks if a number is odd.\n\n',
    'code': 'def is_odd(n: int) -> bool:\n    return True if ',
    'function_name': 'is_odd',
    'unit_tests': [(1, True), (2, False), (0, False), (15, True), (-3, True)],
    'prefills': [
        '',
        '(',
        '0',
        'abs',
        '1',
        '(1',
        '((1',
        '1 +',
        'not',
        'not 1',
        'not 0',
        'bool',
        'bool(',
        'n &',
        '(n &'
    ]
}

In [14]:
from src.functional import generate_with_patch, predict_next_token
from src.tokens import prepare_input

sample = eval_dataset[sample_idx]
instruction = sample["instruction"]
code = sample["code"]
function_name = sample.get("function_name", "func")
unit_tests = sample["unit_tests"]
prefills = sample["prefills"]

prompts = [instruction + "<code>" + code + prefill for prefill in prefills]

inputs = prepare_input(
    prompts=prompts,
    tokenizer=mt,
)


output = generate_with_patch(
    mt=mt,
    inputs=inputs,
    n_gen_per_prompt=1,
    max_new_tokens=64,
    do_sample=False,
    remove_prefix=True,
)

# Run evaluation (returns JSON-like dict)
eval_results = run_evaluation(
    code=code,
    prefills=prefills,
    outputs=output,
    unit_tests=unit_tests,
    function_name=function_name,
    output_stop_string="</code>",
)

# Print results in tabular format
pretty_print_eval_results(eval_results)

"""def is_odd(n: int) -> bool:
    return True if """

                                            Code Completion Results                                             
┏━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┓
┃ Prefill ┃ Logic                        ┃ Verdict ┃ Unit Tests                                         ┃ Logs ┃
┡━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━┩
│ ""      │ " n % 2 != 0 else False"     │ PASSED  │                                                    │      │
├─────────┼──────────────────────────────┼─────────┼────────────────────────────────────────────────────┼──────┤
│ "("     │ "(n % 2) != 0 else False"    │ PASSED  │                                                    │      │
├─────────┼──────────────────────────────┼─────────┼────────────────────────────────────────────────────┼──────┤
│ "0"     │ "0 != n % 2 else False"      │ PASSED  │                                                    │      │
├─────────┼──────────────────────────────┼─────────┼────────────────────────────────────────────────────┼──────┤
│ "abs"   │ "abs(n) % 2 == 1 else False" │ PASSED  │                                                    │      │
├─────────┼──────────────────────────────┼─────────┼────────────────────────────────────────────────────┼──────┤
│ "1"     │ "1 & n else False"           │ PASSED  │                                                    │      │
├─────────┼──────────────────────────────┼─────────┼────────────────────────────────────────────────────┼──────┤
│ "(1"    │ "(1 & n) else False"         │ PASSED  │                                                    │      │
├─────────┼──────────────────────────────┼─────────┼────────────────────────────────────────────────────┼──────┤
│ "((1"   │ "((1 << n) & 1) else False"  │ FAILED  │ is_odd(1)=False, expected True ✗                   │      │
│         │                              │         │ is_odd(2)=False ✓                                  │      │
│         │                              │         │ is_odd(0)=True, expected False ✗                   │      │
│         │                              │         │ is_odd(15)=False, expected True ✗                  │      │
│         │                              │         │ is_odd(-3) raised ValueError: negative shift count │      │
├─────────┼──────────────────────────────┼─────────┼────────────────────────────────────────────────────┼──────┤
│ "1 +"   │ "1 + n % 2 == 0 else False"  │ FAILED  │ is_odd(1)=False, expected True ✗                   │      │
│         │                              │         │ is_odd(2)=False ✓                                  │      │
│         │                              │         │ is_odd(0)=False ✓                                  │      │
│         │                              │         │ is_odd(15)=False, expected True ✗                  │      │
│         │                              │         │ is_odd(-3)=False, expected True ✗                  │      │
├─────────┼──────────────────────────────┼─────────┼────────────────────────────────────────────────────┼──────┤
│ "not"   │ "not n % 2 == 0 else False"  │ PASSED  │                                                    │      │
├─────────┼──────────────────────────────┼─────────┼────────────────────────────────────────────────────┼──────┤
│ "not 1" │ "not 1 & n else False"       │ FAILED  │ is_odd(1)=False, expected True ✗                   │      │
│         │                              │         │ is_odd(2)=True, expected False ✗                   │      │
│         │                              │         │ is_odd(0)=True, expected False ✗                   │      │
│         │                              │         │ is_odd(15)=False, expected True ✗                  │      │
│         │                              │         │ is_odd(-3)=False, expected True ✗                  │      │
├─────────┼──────────────────────────────┼───

Summary: 12/15 passed, 3/15 failed

In [13]:
# full_code = """
# def is_positive(n: int) -> bool:
#     return True if 0 < n else False"""

# evaluate_single_code(full_code, unit_tests, function_name=function_name)

In [14]:
# predict_next_token(
#     mt=mt,
#     inputs=inputs,
#     k=20
# )

In [15]:
from tqdm.auto import tqdm

save_dir = os.path.join(env_utils.DEFAULT_RESULTS_DIR, model_key.split("/")[-1])
os.makedirs(save_dir, exist_ok=True)

for sample_idx, sample in enumerate(tqdm(eval_dataset)):
    instruction = sample["instruction"]
    code = sample["code"]
    function_name = sample.get("function_name", "func")
    unit_tests = sample["unit_tests"]
    prefills = sample["prefills"]

    prompts = [instruction + "<code>" + code + prefill for prefill in prefills]

    inputs = prepare_input(
        prompts=prompts,
        tokenizer=mt,
    )

    output = generate_with_patch(
        mt=mt,
        inputs=inputs,
        n_gen_per_prompt=1,
        max_new_tokens=64,
        do_sample=False,
        remove_prefix=True,
    )

    # Run evaluation (returns JSON-like dict)
    eval_results = run_evaluation(
        code=code,
        prefills=prefills,
        outputs=output,
        unit_tests=unit_tests,
        function_name=function_name,
        output_stop_string="</code>",
    )

    # Save results to JSON file
    save_path = os.path.join(save_dir, f"eval_result_{sample_idx}.json")
    with open(save_path, "w") as f:
        json.dump(eval_results, f, indent=4)
    pretty_print_eval_results(eval_results, output_file=os.path.join(save_dir, f"pretty_print_{sample_idx}.txt"))

    logger.info(f"Saved evaluation results to {save_path}")

  0%|          | 0/14 [00:00<?, ?it/s]

"""def is_odd(n: int) -> bool:
    return True if """

                                    Code Completion Results                                    
┏━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┓
┃ Prefill ┃ Logic                        ┃ Verdict ┃ Unit Tests                        ┃ Logs ┃
┡━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━┩
│ ""      │ " n % 2 == 1 else False"     │ PASSED  │                                   │      │
├─────────┼──────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ "("     │ "(n % 2 == 1) else False"    │ PASSED  │                                   │      │
├─────────┼──────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ "0"     │ "0 != n % 2 else False"      │ PASSED  │                                   │      │
├─────────┼──────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ "abs"   │ "abs(n) % 2 == 1 else False" │ PASSED  │                                   │      │
├─────────┼──────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ "1"     │ "1 == n % 2 else False"      │ PASSED  │                                   │      │
├─────────┼──────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ "(1"    │ "(1 & n) else False"         │ PASSED  │                                   │      │
├─────────┼──────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ "((1"   │ "((1 & n) == 1) else False"  │ PASSED  │                                   │      │
├─────────┼──────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ "1+"    │ "1+1 == 3 else False"        │ FAILED  │ is_odd(1)=False, expected True ✗  │      │
│         │                              │         │ is_odd(2)=False ✓                 │      │
│         │                              │         │ is_odd(0)=False ✓                 │      │
│         │                              │         │ is_odd(15)=False, expected True ✗ │      │
│         │                              │         │ is_odd(-3)=False, expected True ✗ │      │
├─────────┼──────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ "not"   │ "not n % 2 == 0 else False"  │ PASSED  │                                   │      │
├─────────┼──────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ "not 1" │ "not 1 & n else False"       │ FAILED  │ is_odd(1)=False, expected True ✗  │      │
│         │                              │         │ is_odd(2)=True, expected False ✗  │      │
│         │                              │         │ is_odd(0)=True, expected False ✗  │      │
│         │                              │         │ is_odd(15)=False, expected True ✗ │      │
│         │                              │         │ is_odd(-3)=False, expected True ✗ │      │
├─────────┼──────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ "not 0" │ "not 0 == n % 2 else False"  │ PASSED  │                                   │      │
├─────────┼──────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ "bool"  │ "bool(n % 2) else False"     │ PASSED  │                                   │      │
├─────────┼──────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ "bool(" │ "bool(1 & n) else False"     │ PASSED  │                                   │      │
├─────────┼──────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ "n &"   │ "n & 1 else False"           │ PASSED  │                                   │      │
├─────────┼──────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ "(n &"  │ "(n & 1) == 1 else False"    │ PASSED  │                                   │      │
└─────────┴──────────────────────────────┴─────────┴────────────

Summary: 13/15 passed, 2/15 failed

2026-01-23 17:30:41 __main__ INFO     Saved evaluation results to /disk/u/arnab/Codes/Projects/goals/results/Qwen3-Coder-30B-A3B-Instruct/eval_result_0.json


  7%|▋         | 1/14 [00:59<12:47, 59.04s/it]

... (61 outputs left out)

                                                    Code Completion Results                                                    
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┓
┃ Prefill     ┃ Logic                                          ┃ Verdict ┃ Unit Tests                                  ┃ Logs ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━┩
│ ""          │ "1 == len(set(lst)) else False"                │ PASSED  │                                             │      │
├─────────────┼────────────────────────────────────────────────┼─────────┼─────────────────────────────────────────────┼──────┤
│ "len"       │ "len(set(lst)) == 1 else False"                │ PASSED  │                                             │      │
├─────────────┼────────────────────────────────────────────────┼─────────┼─────────────────────────────────────────────┼──────┤
│ "len("      │ "len( set(lst) ) == 1 else False"              │ PASSED  │                                             │      │
├─────────────┼────────────────────────────────────────────────┼─────────┼─────────────────────────────────────────────┼──────┤
│ "set"       │ "set(lst) == 1 else False"                     │ FAILED  │ all_same([1])=False, expected True ✗        │      │
│             │                                                │         │ all_same([1, 1, 1])=False, expected True ✗  │      │
│             │                                                │         │ all_same([1, 2])=False ✓                    │      │
│             │                                                │         │ all_same(['a', 'a'])=False, expected True ✗ │      │
│             │                                                │         │ all_same([1, 1, 2])=False ✓                 │      │
├─────────────┼────────────────────────────────────────────────┼─────────┼─────────────────────────────────────────────┼──────┤
│ "set("      │ "set(1 for x in lst) == {1} else False"        │ FAILED  │ all_same([1])=True ✓                        │      │
│             │                                                │         │ all_same([1, 1, 1])=True ✓                  │      │
│             │                                                │         │ all_same([1, 2])=True, expected False ✗     │      │
│             │                                                │         │ all_same(['a', 'a'])=True ✓                 │      │
│             │                                                │         │ all_same([1, 1, 2])=True, expected False ✗  │      │
├─────────────┼────────────────────────────────────────────────┼─────────┼─────────────────────────────────────────────┼──────┤
│ "all"       │ "all(x == lst[0] for x in lst) else False"     │ PASSED  │                                             │      │
├─────────────┼────────────────────────────────────────────────┼─────────┼─────────────────────────────────────────────┼──────┤
│ "all("      │ "all(elt == lst[0] for elt in lst) else False" │ PASSED  │                                             │      │
├─────────────┼────────────────────────────────────────────────┼─────────┼─────────────────────────────────────────────┼──────┤
│ "lst[0]"    │ "lst[0] == lst[-1] else False"                 │ PASSED  │                                             │      │
├─────────────┼────────────────────────────────────────────────┼─────────┼─────────────────────────────────────────────┼──────┤
│ "lst.count" │ "lst.count(lst[0]) == len(lst) else False"     │ PASSED  │                                             │      │
└─────────────┴────────────────────────────────────────────────┴─────────┴─────────────────────────────────────────────┴──────┘

Summary: 7/9 passed, 2/9 failed

2026-01-23 17:43:46 __main__ INFO     Saved evaluation results to /disk/u/arnab/Codes/Projects/goals/results/Qwen3-Coder-30B-A3B-Instruct/eval_result_13.json


100%|██████████| 14/14 [14:04<00:00, 60.29s/it]


In [16]:

# with open(os.path.join(save_dir, "eval_result_3.json"), "r") as f:
#     eval_demo = json.load(f) 

# pretty_print_eval_results(eval_demo, output_file="results.txt")